In [70]:
import cv2
import numpy as np
from paddleocr import PaddleOCR
from ollama import chat
import os
from os import listdir

c:\python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (1.26.16) or chardet (7.6.0)/charset_normalizer (3.1.0) doesn't match a supported version!
  warnings.warn(


Supporting Functions

functions to crop a series of images

In [82]:
def get_horizontal_crops(im, breaks, height, width):
    images = []
    for i in range(len(breaks)):
        if i == 0:
            cropped_img = im[0:breaks[i], 0:width]
            images.append(cropped_img)
        elif i == len(breaks):
            cropped_img = im[breaks[i]:height, 0:width]
            images.append(cropped_img)
        else:
            cropped_img = im[breaks[i-1]:breaks[i], 0:width]
            images.append(cropped_img)
    return images

def get_vertical_crops(im, breaks, height, width):
    images = []
    for i in range(len(breaks)):
        if i == 0:
            cropped_img = im[0:height, 0:breaks[i]]
            images.append(cropped_img)
        elif i == len(breaks):
            cropped_img = im[0:height, breaks[i]:width]
            images.append(cropped_img)
        else:
            cropped_img = im[0:height, breaks[i-1]:breaks[i]]
            images.append(cropped_img)
    return images

function to find breakpoints in an image based on the horizontal projection to inform cropping

In [22]:
def find_break(hoz_prj,empty_rows):
    zeros = 0
    splitx = []
    row = 0
    detected=False
    for i in hoz_prj:
        if row == 0 and i == 0:
            detected = True
            zeros = zeros + 1
        elif i == 0:
            zeros = zeros + 1
        if zeros == empty_rows and detected == False:
            splitx.append(row)
            detected=True
            zeros = 0
        elif detected == True and i != 0:
            detected = False
            zeros = 0
        row = row+1
    return splitx   

function to process the image and call functions to find breaks, returns list of break points for cropping

In [93]:
def find_breaks(img_link):
    im = cv2.imread(img_link, cv2.IMREAD_GRAYSCALE)
    height, width = im.shape
    GaussianFilter= cv2.GaussianBlur(im, (5,5), 0)
    thret, binarizedImage = cv2.threshold(GaussianFilter, 220, 255, cv2.THRESH_BINARY)
    binarizedImage[binarizedImage == 0] = 1
    binarizedImage[binarizedImage == 255] = 0
    horizontal_projection = np.sum(binarizedImage, axis=1)
    vertical_projection = np.sum(binarizedImage, axis=0)
    empty_rows = 5 #hardcoded in for now but we could build logic to determine this
    horizontal_breaks = find_break(horizontal_projection,empty_rows)
    vertical_breaks = find_break(vertical_projection,empty_rows)
    return im, horizontal_breaks, vertical_breaks, height, width

In [94]:
def process_images(folder_link,output_link):
    # for image in folder
    i = 0
    for item in os.listdir(folder_link):
        link = folder_link+"/"+item
        #filter to only let images in because the first folder item is sometimes: captures/.ipynb_checkpoints
        if (link.endswith(".JPG")):
            i=i+1
            j=0
            im, horz,vert,height, width = find_breaks(link)
            # crop images vert then horz
            vert_images = get_vertical_crops(im, vert, height, width)
            cropped_images = []
            for vert_im in vert_images:
                horiz_images = get_horizontal_crops(vert_im, horz,height, width)
                for horiz_im in horiz_images:
                    j=j+1
                    cropped_images.append(horiz_im)
                    # save cropped images in a folder for each parent image
                    cv2.imwrite(output_link+'/image'+str(i)+'crop'+str(j)+'.png', horiz_im)
    
    # run ocr against cropped images
    # use LLM to check ocr output

In [95]:
folder_link = "captures"
output_link = "output/Books"
process_images(folder_link,output_link)

[314]
[]


In [47]:
pa(folder_link,"")

captures/.ipynb_checkpoints
captures/Alice.JPG
captures/Book2.JPG


In [71]:
ocr = PaddleOCR(
    lang='en', 
    use_textline_orientation=True, 
    enable_mkldnn=False
)

c:\python310\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\steph\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\steph\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\steph\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)

In [89]:
def runOCR(folder_link):
    outputs = []
    for item in os.listdir(folder_link):
            # find breaks
            link = folder_link+"/"+item
            if (link.endswith(".png")):
                OCR_text = ocr.predict(link)
                # within each image the OCR engine will split text into sections, so we will join them back into one block of text per image
                OCR_text = ' '.join(OCR_text[0]["rec_texts"])
                im_dict = {
                  "file_name": link,
                  "ocr_text": OCR_text,
                  "llm_text": ""
                }
                outputs = outputs.append(im_dict)
    

In [86]:
from ollama import chat
from pydantic import BaseModel

class PageText(BaseModel):
  correctedText: str
  explanation: str

def LLM_Review_JSON (raw_ocr_text):
    structured_data = chat(
        model="llama3.2:3b",  # Must match the exact model name downloaded in Ollama
        messages=[
            {
                "role": "system", 
                "content": (
                    "You are an LLM Supervisor reviewing raw, messy OCR text. Your job is to:\n"
                    "1. Fix obvious OCR typographical errors (e.g., 'To1al' -> 'Total', '8ump' -> 'Bump', 'buv' -> 'buy').\n"
                    "2. Return the whole corrected text, do not split into sections"
                    "3. Correct words that appear contextually incorrect (e.g. 'he car' -> 'the car')"
                )
            },
            {"role": "user", "content": f"Here is the raw OCR output:\n\n{raw_ocr_text}"}
        ],format=PageText.model_json_schema(),
    )
    return structured_data


In [67]:
text = """Alice was beginning to get very tired of sitting by her sister on the bank. and of having nothing to dor once or twice she had peeped into the book he sister was reading, but it had no pictures or conversations in it, and what is the use of a book.' thought Alice "without pictures or conversation? so she was considering in her own mind (as well as she could, for the hot ay made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and pickin he daisies, when suddenly a White Rabbit with pink eves ran close by her"""
text = text.replace("\"", "'")
LLM_text = LLM_Review_JSON(text)

In [68]:
outputs = PageText.model_validate_json(LLM_text.message.content)
print(outputs.correctedText)

Alice was beginning to get very tired of sitting by her sister on the bank. And of having nothing to do, once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, and what is the use of a book.' thought Alice 'without pictures or conversation? so she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies, when suddenly a White Rabbit with pink eyes ran close by her
